# 21｜Transformer加速_推論與參數高效微調

量化 KV cache 的計算差異，並計算 LoRA 的可訓練參數量。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/21_Transformer加速_推論與參數高效微調.md`，目前 38 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

KV 等價、GQA/分塊 softmax、推測驗證、LoRA、MoE、packing/梯度累積。

各格會標示實驗資料、評估方式與適用範圍；架構圖用於說明機制與張量形狀。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 21-03 | 5、6、7、25、26 | 課堂小實驗 |
| 21-04 | 3、5、6 | 執行結果圖 |
| 21-05 | 6、7、8、9、13、14、15、16、17、18、19、20、21、22、38 | 1. KV cache 等價性、GQA 與分塊 Softmax |
| 21-06 | 10、11、12、23、24、25、26、27、28、38 | 2. 推測驗證、低秩更新與稀疏專家 |
| 21-07 | 29、30、31、32、33、34、35、36、37、38 | 3. Packing、文件邊界與尾組梯度累積 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 21-01｜投影片 21：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 21-02｜投影片 21：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 課堂小實驗

此圖只計算重複 K/V 投影的 token 次數，不是整個 attention FLOPs、GPU 記憶體峰值或實測延遲。

In [ ]:
# 程式 21-03｜投影片 21：5、6、7、25、26 頁
lengths = np.array([16, 32, 64, 128, 256])
without_cache = lengths * (lengths + 1) / 2
with_cache = lengths
d_in = d_out = 4096; rank = 8
full = d_in * d_out
lora = rank * (d_in + d_out)
table = pd.DataFrame({'sequence_length': lengths,
                      'projected_token_states_without_cache': without_cache.astype(int),
                      'with_KV_cache': with_cache,
                      'ratio': without_cache / with_cache})
print(f'LoRA trainable parameters: {lora:,} vs full matrix {full:,} ({lora/full:.3%})')
table

## 執行結果圖

In [ ]:
# 程式 21-04｜投影片 21：3、5、6 頁
plt.loglog(lengths, without_cache, 'o-', label='recompute prefix')
plt.loglog(lengths, with_cache, 's-', label='KV cache')
plt.xlabel('generated sequence length'); plt.ylabel('projected token states')
plt.title('KV cache avoids repeated key/value projections'); plt.legend()
savefig('21_transformer_efficiency_demo')
report('transformer_efficiency', {'lora_params': lora, 'full_params': full, 'table': table.to_dict('records')})

## 1. KV cache 等價性、GQA 與分塊 Softmax

小型 NumPy 逐 token 比較有無 cache 的輸出，再驗證穩定的分塊 Softmax。這驗證數學等價，不模擬 FlashAttention 的 GPU I/O 或效能。

In [ ]:
# 程式 21-05｜投影片 21：6、7、8、9、13、14、15、16、17、18、19、20、21、22、38 頁
rng21=np.random.default_rng(SEED);sequence21=rng21.normal(size=(12,8));Wq21,Wk21,Wv21=[rng21.normal(size=(8,8)) for _ in range(3)]
def softmax21(z):
    e=np.exp(z-z.max(-1,keepdims=True));return e/e.sum(-1,keepdims=True)
cache_k21=[];cache_v21=[];cached21=[];reference21=[]
for t21,x21 in enumerate(sequence21):
    q21=x21@Wq21;cache_k21.append(x21@Wk21);cache_v21.append(x21@Wv21)
    cached21.append(softmax21(q21@np.array(cache_k21).T/np.sqrt(8))@np.array(cache_v21))
    k21=sequence21[:t21+1]@Wk21;v21=sequence21[:t21+1]@Wv21
    reference21.append(softmax21(q21@k21.T/np.sqrt(8))@v21)
assert np.allclose(cached21,reference21)
assert 2*1*4096*32*8*128*2==536870912
print('KV example MiB:',2*1*4096*32*8*128*2/2**20)
q21=rng21.normal(size=(4,5,2));k_small21=rng21.normal(size=(2,5,2));v_small21=rng21.normal(size=(2,5,2))
k_group21=np.repeat(k_small21,2,axis=0);v_group21=np.repeat(v_small21,2,axis=0)
gqa21=softmax21(q21@k_group21.transpose(0,2,1)/np.sqrt(2))@v_group21
print('GQA Q / compact KV / output:',q21.shape,k_small21.shape,gqa21.shape)
# 線上 log-sum-exp 合併不同分塊，無需存整列權重
scores21=rng21.normal(0,20,31);values21=rng21.normal(size=(31,4));max21=-np.inf;den21=0.;num21=np.zeros(4)
for a21 in range(0,len(scores21),7):
    block21=scores21[a21:a21+7];newmax21=max(max21,block21.max());rescale21=np.exp(max21-newmax21)
    exp21=np.exp(block21-newmax21);num21=num21*rescale21+exp21@values21[a21:a21+7];den21=den21*rescale21+exp21.sum();max21=newmax21
assert np.allclose(num21/den21,softmax21(scores21)@values21)
# Performer 類核重排機制：phi=ELU+1，不冒稱是原 FAVOR+ 隨機特徵
Q21=rng21.normal(size=(5,4));K21=rng21.normal(size=(6,4));V21=rng21.normal(size=(6,3))
phi21=lambda z:np.where(z>0,z+1,np.exp(z))
qphi21,kphi21=phi21(Q21),phi21(K21);kernel21=qphi21@kphi21.T
fast21=(qphi21@(kphi21.T@V21))/(qphi21@kphi21.sum(0))[:,None]
assert np.allclose(fast21,(kernel21/kernel21.sum(1,keepdims=True))@V21)
mask21=np.abs(np.arange(12)[:,None]-np.arange(12)[None,:])<=2
plt.imshow(mask21,cmap='binary');plt.xlabel('key');plt.ylabel('query');plt.title('Local sparse attention (bidirectional example)');savefig('c21_sparse_mask')

## 2. 推測驗證、低秩更新與稀疏專家

抽樣式推測不能只比較 argmax；接受率與拒絕後殘差分布必須配套。LoRA 凍結 W，只更新 A/B；下例以 W 的 [輸出,輸入] 慣例對齊投影片。

In [ ]:
# 程式 21-06｜投影片 21：10、11、12、23、24、25、26、27、28、38 頁
target21=np.array([.6,.3,.1]);draft21=np.array([.3,.4,.3]);accept21=np.minimum(1,target21/draft21)
accepted_mass21=draft21*accept21;residual21=np.maximum(target21-draft21,0);residual21/=residual21.sum()
reconstructed21=accepted_mass21+(1-accepted_mass21.sum())*residual21
assert np.allclose(reconstructed21,target21)
print('Speculative marginal equals target:',reconstructed21)
target_greedy21=[2,1,3];draft_greedy21=[2,0,4];accepted_prefix21=[]
for a21,b21 in zip(target_greedy21,draft_greedy21):
    if a21!=b21:break
    accepted_prefix21.append(a21)
print('Greedy accepted prefix:',accepted_prefix21,'then use target token at mismatch')
W21=rng21.normal(size=(6,8));W_copy21=W21.copy();A21=rng21.normal(0,.1,(2,8));B21=np.zeros((6,2));inputs21=rng21.normal(size=(50,8))
truth21=inputs21@(W21+.15*rng21.normal(size=W21.shape)).T;losses21=[]
for _ in range(120):
    hidden21=inputs21@A21.T;pred21=inputs21@W21.T+hidden21@B21.T;error21=(pred21-truth21)/len(inputs21)
    losses21.append(.5*np.sum((pred21-truth21)**2)/len(inputs21))
    dB21=error21.T@hidden21;dA21=(error21@B21).T@inputs21;B21-=.1*dB21;A21-=.1*dA21
assert np.array_equal(W21,W_copy21) and losses21[-1]<losses21[0]
assert np.allclose(inputs21@(W21+B21@A21).T,inputs21@W21.T+(inputs21@A21.T)@B21.T)
plt.plot(losses21);plt.xlabel('LoRA update');plt.ylabel('synthetic regression loss');savefig('c21_lora_training')
# top-1 MoE 只執行被指派的專家；回報負載，而非聲稱均衡
router21=softmax21(inputs21[:10]@rng21.normal(size=(8,3)));chosen21=router21.argmax(1)
experts21=rng21.normal(size=(3,8,4));expert_out21=np.stack([inputs21[i]@experts21[e] for i,e in enumerate(chosen21)])
print('Expert loads / output shape:',np.bincount(chosen21,minlength=3),expert_out21.shape)
# Adapter：瓶頸殘差而非直接合併成 BA
adapter21=inputs21+np.maximum(inputs21@rng21.normal(size=(8,2)),0)@rng21.normal(size=(2,8));assert adapter21.shape==inputs21.shape

## 3. Packing、文件邊界與尾組梯度累積

打包後同一文件內維持 causal mask，跨文件不得關注。不同大小 microbatch 以樣本數加權，避免最後不足一組被少算或放大。

In [ ]:
# 程式 21-07｜投影片 21：29、30、31、32、33、34、35、36、37、38 頁
lengths21=[3,5,2];docids21=np.repeat(np.arange(3),lengths21);positions21=np.arange(len(docids21))
packing21=(docids21[:,None]==docids21[None,:])&(positions21[:,None]>=positions21[None,:])
assert not packing21[5,0]
plt.imshow(packing21,cmap='binary');plt.xlabel('key in packed sequence');plt.ylabel('query');plt.title('Document-aware causal packing');savefig('c21_packing_mask')
print('Naive padding / packed tokens:',len(lengths21)*max(lengths21),sum(lengths21))
xacc21=rng21.normal(size=(10,3));yacc21=rng21.normal(size=10);wacc21=np.zeros(3)
full_grad21=xacc21.T@(xacc21@wacc21-yacc21)/len(yacc21);acc21=np.zeros(3)
for index21 in [np.arange(4),np.arange(4,8),np.arange(8,10)]:
    batch_grad21=xacc21[index21].T@(xacc21[index21]@wacc21-yacc21[index21])/len(index21)
    acc21+=batch_grad21*len(index21)/len(yacc21)
assert np.allclose(acc21,full_grad21)
# Checkpoint 概念：重算中間值可取得相同梯度，增加前向計算
z21=xacc21@wacc21;stored_grad21=xacc21.T@(z21-yacc21)/10
recomputed_grad21=xacc21.T@(xacc21@wacc21-yacc21)/10
assert np.allclose(stored_grad21,recomputed_grad21)
print('Weighted accumulation gradient:',acc21,'data-parallel weighted sum has the same algebra')

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。